# บทที่ 3 การทำความสะอาดและการเตรียมลักษณะข้อมูลฟิสิกส์
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch03_cleaning/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch03_cleaning/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch03_cleaning/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch03_cleaning'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'matplotlib': 'matplotlib', 'nbclient': 'nbclient>=0.8', 'nbformat': 'nbformat>=5.9', 'numpy': 'numpy', 'pandas': 'pandas', 'pint': 'pint>=0.22', 'scipy': 'scipy'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_make_raw_data.py`

Ch.3  Create the SYNTHETIC raw data used by every script of this chapter.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_make_raw_data.py  จากโฟลเดอร์ของบท
__file__ = '01_make_raw_data.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Create the SYNTHETIC raw data used by every script of this chapter.

The book refers to files such as data/raw/galaxy10_metadata.csv and data/raw/afm_spectrum.csv.
They are teaching files built here with a fixed seed; the numbers are NOT measurements.
  * galaxy table: Galaxy10 DECaLS class names, but redshift and magnitudes are simulated.
  * AFM files: simulated curves (the author's own AFM measurements are not published here).
  * 4-lepton events: simulated H -> ZZ* -> 4l signal plus a smooth background.

Output (data/raw/)
  galaxy10_metadata.csv      2,000 galaxies: image_id, galaxy_class, redshift, r_mag, g_mag
  galaxy10_image_index.csv   image_id -> image file (some ids missing, some extra)
  galaxy10_truth.csv         the true redshift of every galaxy (answer key, for E3.4 only)
  galaxy10_messy.csv         ~500 rows with NaN, -999 sentinels and outliers (for E3.2)
  galaxy_images.npy          4 synthetic images, shape (4, 256, 256, 3), 2 hot pixels in image 0
  afm_timeseries.csv         force signal while scanning: drift + spikes + noise
  afm_spectrum.csv           one force-distance curve (approach + retract) with baseline shift
  afm_curves.csv             15 force-distance curves with different stiffness/adhesion (E3.5)
  afm_curves_truth.csv       the parameters used to make them (answer key)
  higgs_4l_events.csv        E, px, py, pz of four leptons per event [GeV]
"""
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 2026
rng = np.random.default_rng(SEED)
RAW = Path(__file__).resolve().parent / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

CLASSES = ["Disturbed", "Merging", "Round Smooth", "In-between Round Smooth",
           "Cigar Shaped Smooth", "Barred Spiral", "Unbarred Tight Spiral",
           "Unbarred Loose Spiral", "Edge-on without Bulge", "Edge-on with Bulge"]


# --------------------------------------------------------------- galaxies
def make_galaxies(n=2000):
    cls = rng.choice(len(CLASSES), n, p=[.06, .10, .15, .15, .06, .11, .10, .15, .06, .06])
    z_true = np.clip(rng.gamma(3.0, 0.035, n), 0.005, 0.40)
    # toy magnitude-redshift relation (inverse of photo_z_estimate in the book) + scatter
    r_mag = 15.0 + z_true / 0.05 + rng.normal(0, 0.6, n)
    color = np.array([0.7, 0.7, 0.9, 0.85, 0.8, 0.75, 0.6, 0.55, 0.65, 0.8])[cls]
    g_mag = r_mag + color + rng.normal(0, 0.1, n)
    # redshift missing more often for faint galaxies (MAR on r_mag), ~15 % overall
    p_miss = 1 / (1 + np.exp(-(r_mag - 18.9) / 0.6))
    missing = rng.random(n) < p_miss
    df = pd.DataFrame({
        "image_id": [f"galaxy_{i:05d}" for i in range(n)],
        "galaxy_class": [CLASSES[c] for c in cls],
        "redshift": np.where(missing, np.nan, np.round(z_true, 4)),
        "r_mag": np.round(r_mag, 3),
        "g_mag": np.round(g_mag, 3).astype(object),
    })
    # five bright foreground stars that slipped into the catalogue (r < 8)
    stars = rng.choice(n, 5, replace=False)
    df.loc[stars, "r_mag"] = np.round(rng.uniform(6.0, 7.8, 5), 3)
    df.loc[stars, "redshift"] = 0.0
    # three g_mag values typed as text in the source file -> the column is read as text
    bad = rng.choice(np.setdiff1d(np.arange(n), stars), 3, replace=False)
    df.loc[bad, "g_mag"] = "--"
    df.to_csv(RAW / "galaxy10_metadata.csv", index=False)
    pd.DataFrame({"image_id": df["image_id"], "z_true": np.round(z_true, 4),
                  "is_star": np.isin(np.arange(n), stars)}).to_csv(RAW / "galaxy10_truth.csv",
                                                                   index=False)

    # image index: 20 galaxies have no image file, 15 image files have no metadata row
    keep = np.sort(rng.choice(n, n - 20, replace=False))
    ids = [f"galaxy_{i:05d}" for i in keep] + [f"galaxy_{i:05d}" for i in range(n, n + 15)]
    pd.DataFrame({"image_id": ids,
                  "image_file": [f"images/{i}.png" for i in ids]}).to_csv(
        RAW / "galaxy10_image_index.csv", index=False)
    return df, z_true


def make_messy(df, n=500):
    m = df.sample(n, random_state=SEED).reset_index(drop=True).copy()
    m["g_mag"] = pd.to_numeric(m["g_mag"], errors="coerce")
    idx = rng.permutation(n)
    m.loc[idx[:20], "redshift"] = -999          # sentinel: "could not be measured"
    m.loc[idx[20:25], "r_mag"] = -999
    m.loc[idx[25:31], "r_mag"] = np.round(rng.uniform(25.5, 28.0, 6), 3)   # too faint
    m.loc[idx[31:33], "redshift"] = np.round(-rng.uniform(0.01, 0.05, 2), 4)  # sign typo
    m.to_csv(RAW / "galaxy10_messy.csv", index=False)


def make_images(n=4, size=256):
    y, x = np.mgrid[0:size, 0:size]
    imgs = np.zeros((n, size, size, 3), dtype=np.float32)
    for k in range(n):
        x0, y0 = 128 + rng.normal(0, 3, 2)
        q, pa = rng.uniform(0.5, 1.0), rng.uniform(0, np.pi)
        xr = (x - x0) * np.cos(pa) + (y - y0) * np.sin(pa)
        yr = -(x - x0) * np.sin(pa) + (y - y0) * np.cos(pa)
        r = np.hypot(xr, yr / q)
        for b, amp in enumerate((0.6, 1.0, 1.3)):                  # g, r, z bands
            bulge = amp * 60 * np.exp(-7.67 * ((r / 6) ** 0.25 - 1)) / np.exp(7.67)
            disk = amp * 8 * np.exp(-r / 25)
            imgs[k, :, :, b] = bulge + disk + rng.normal(0, 0.5, (size, size))
    imgs[0, 40, 200, 0] = 60.0        # two hot pixels in image 0, band 0
    imgs[0, 210, 35, 0] = 45.0
    np.save(RAW / "galaxy_images.npy", imgs)


# --------------------------------------------------------------- AFM
def afm_curve(k_sample, adhesion_nN, z_contact=20.0, snap=15.0, offset=0.8, noise=0.05,
              n=301):
    """One approach + retract force-distance curve [nN vs nm].
    distance_nm is the piezo height above the contact point + z_contact."""
    z = np.linspace(300.0, 0.0, n)
    appr = np.where(z < z_contact, k_sample * (z_contact - z), 0.0)
    z_r = z[::-1]
    retr = np.where(z_r < z_contact, k_sample * (z_contact - z_r), 0.0)
    # adhesion: the tip stays stuck and is pulled down to -adhesion before snapping off
    stick = (z_r >= z_contact) & (z_r <= z_contact + snap)
    retr = np.where(stick, -adhesion_nN * (z_r - z_contact) / snap, retr)
    d = np.concatenate([z, z_r])
    f = np.concatenate([appr, retr]) + offset + rng.normal(0, noise, 2 * n)
    seg = ["approach"] * n + ["retract"] * n
    return pd.DataFrame({"segment": seg, "distance_nm": np.round(d, 3),
                         "force_nN": np.round(f, 4)})


def make_afm():
    # time series: 2 s at 1 kHz while the tip scans a surface
    t = np.arange(0, 2.0, 0.001)
    topo = 0.3 * np.sin(2 * np.pi * 1.5 * t)
    drift = 0.4 * t                                  # slow thermal drift [nN/s]
    f = 2.0 + topo + drift + rng.normal(0, 0.03, t.size)
    spikes = rng.choice(t.size, 15, replace=False)
    f[spikes] += rng.choice([-1, 1], 15) * rng.uniform(1.0, 2.5, 15)
    pd.DataFrame({"time_s": np.round(t, 4), "force_nN": np.round(f, 4)}).to_csv(
        RAW / "afm_timeseries.csv", index=False)
    pd.DataFrame({"spike_index": np.sort(spikes)}).to_csv(RAW / "afm_timeseries_spikes.csv",
                                                           index=False)

    afm_curve(k_sample=0.25, adhesion_nN=3.0).to_csv(RAW / "afm_spectrum.csv", index=False)

    rows, truth = [], []
    for cid in range(15):
        k_s = rng.uniform(0.1, 0.5)                  # nN/nm
        adh = rng.uniform(1.0, 5.0)                  # nN
        off = rng.normal(0.5, 0.3)
        c = afm_curve(k_s, adh, offset=off)
        c.insert(0, "curve_id", cid)
        rows.append(c)
        truth.append({"curve_id": cid, "slope_nN_per_nm": k_s, "adhesion_nN": adh,
                      "work_aJ": 0.5 * adh * 15.0, "baseline_nN": off})
    pd.concat(rows).to_csv(RAW / "afm_curves.csv", index=False)
    pd.DataFrame(truth).round(4).to_csv(RAW / "afm_curves_truth.csv", index=False)


# --------------------------------------------------------------- 4-lepton events
def two_body(M, m1, m2, n):
    """Momenta of two daughters in the parent rest frame, isotropic."""
    p = np.sqrt((M**2 - (m1 + m2)**2) * (M**2 - (m1 - m2)**2)) / (2 * M)
    cth = rng.uniform(-1, 1, n); phi = rng.uniform(0, 2 * np.pi, n)
    sth = np.sqrt(1 - cth**2)
    v = np.stack([p * sth * np.cos(phi), p * sth * np.sin(phi), p * cth], axis=1)
    E1 = np.sqrt(p**2 + m1**2); E2 = np.sqrt(p**2 + m2**2)
    return np.column_stack([E1, v]), np.column_stack([E2, -v])


def boost(P, beta):
    """Lorentz boost of four-vectors P (n,4) by velocity vectors beta (n,3)."""
    b2 = np.sum(beta**2, axis=1)
    g = 1 / np.sqrt(1 - b2)
    bp = np.sum(beta * P[:, 1:], axis=1)
    E = g * (P[:, 0] + bp)
    coef = np.where(b2 > 0, (g - 1) * bp / np.where(b2 > 0, b2, 1), 0.0) + g * P[:, 0]
    return np.column_stack([E, P[:, 1:] + coef[:, None] * beta])


def make_4l(n_sig=400, n_bkg=1600):
    M = np.concatenate([rng.normal(125.0, 1.8, n_sig),              # Higgs + resolution
                        70 + rng.gamma(2.5, 45.0, n_bkg)])            # smooth background
    n = M.size
    mZ1 = np.minimum(rng.normal(91.19, 2.5, n), M - 13)
    mZ2 = np.clip(rng.uniform(12, 60, n), 12, M - mZ1 - 1)
    Z1, Z2 = two_body(M, mZ1, mZ2, n)
    leps = []
    for Z, mZ in ((Z1, mZ1), (Z2, mZ2)):
        l1, l2 = two_body(mZ, 0.0, 0.0, n)
        beta = Z[:, 1:] / Z[:, [0]]
        leps += [boost(l1, beta), boost(l2, beta)]
    # boost the whole system along the beam (z) and give it a little transverse kick
    beta_lab = np.column_stack([rng.normal(0, 0.05, n), rng.normal(0, 0.05, n),
                                rng.uniform(-0.7, 0.7, n)])
    leps = [boost(l, beta_lab) for l in leps]
    order = rng.permutation(n)
    cols = {}
    for i, l in enumerate(leps, start=1):
        for j, c in enumerate(("E", "px", "py", "pz")):
            cols[f"{c}{i}"] = np.round(l[order, j], 4)
    ev = pd.DataFrame(cols)
    ev.insert(0, "is_signal", (np.arange(n) < n_sig)[order])
    ev.to_csv(RAW / "higgs_4l_events.csv", index=False)


if __name__ == "__main__":
    df, _ = make_galaxies()
    make_messy(df)
    make_images()
    make_afm()
    make_4l()
    for p in sorted(RAW.iterdir()):
        print(f"{p.name:30s} {p.stat().st_size / 1024:8.1f} kB")

## `02_tabular_missing.py` — โค้ดที่ 3.1, 3.2

Ch.3  Count missing values, mark instead of delete, and ask WHY values are missing.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_tabular_missing.py  จากโฟลเดอร์ของบท
__file__ = '02_tabular_missing.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Count missing values, mark instead of delete, and ask WHY values are missing.

Book references: ssec:tabular, code:tabular-missing, code:tabular-mask
Text: "suppose redshift is missing for about 15 %" -> in this synthetic table 15.3 %.
The table is SYNTHETIC (made by 01_make_raw_data.py); redshifts are missing more often
for faint galaxies, i.e. not completely at random.
"""
import os
from pathlib import Path

os.chdir(Path(__file__).resolve().parent)      # so the book's relative paths work

### โค้ดที่ 3.1

In [ ]:
# --- code:tabular-missing ---------------------------------------------------
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/galaxy10_metadata.csv')
print(df.shape)              # (rows, columns)
print(df.isna().sum())       # number of NaN in each column

### โค้ดที่ 3.2

In [ ]:
# --- code:tabular-mask ------------------------------------------------------
df['has_redshift'] = df['redshift'].notna()
print(f"with redshift: {df['has_redshift'].sum()} / {len(df)}")
print(f"missing fraction: {100 * (1 - df['has_redshift'].mean()):.1f} %")

# --- is the missingness random? compare brightness of the two groups --------
r = df['r_mag']
print("\nmedian r_mag with redshift   :", round(r[df['has_redshift']].median(), 2))
print("median r_mag without redshift:", round(r[~df['has_redshift']].median(), 2))
bins = pd.cut(r, [0, 8, 16, 17, 18, 19, 20, 30])
print("\nmissing fraction by r_mag bin (larger r_mag = fainter):")
print((1 - df.groupby(bins, observed=True)['has_redshift'].mean()).round(3).to_string())
print("-> missing values concentrate in faint galaxies: NOT missing completely at random,"
      " so dropping those rows biases the sample toward bright galaxies")

# --- the r < 8 check from the text ------------------------------------------
too_bright = df[df['r_mag'] < 8]
print(f"\nobjects with r_mag < 8 (probably foreground stars): {len(too_bright)}")
print(too_bright[['image_id', 'r_mag', 'redshift']].to_string(index=False))

## `03_timeseries_afm.py` — โค้ดที่ 3.3

Ch.3  AFM force time series: remove spikes with a median filter, then remove drift.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_timeseries_afm.py  จากโฟลเดอร์ของบท
__file__ = '03_timeseries_afm.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  AFM force time series: remove spikes with a median filter, then remove drift.
Also: why aliasing cannot be undone afterwards (misconception box, review question 4).

Book references: ssec:timeseries, code:ts-clean
The AFM signal is SYNTHETIC: 2 s at 1 kHz, 1.5 Hz topography, linear drift, 15 spikes.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent)
afm = pd.read_csv('data/raw/afm_timeseries.csv')
spikes = pd.read_csv('data/raw/afm_timeseries_spikes.csv')['spike_index'].values

### โค้ดที่ 3.3

In [ ]:
# --- code:ts-clean ----------------------------------------------------------
from scipy.signal import medfilt, detrend

afm['force_clean'] = medfilt(afm['force_nN'].values, kernel_size=5)
afm['force_detrended'] = detrend(afm['force_clean'].values, type='linear')

# --- checks -----------------------------------------------------------------
raw, clean = afm['force_nN'].values, afm['force_clean'].values
resid = raw - clean
print(f"{len(afm)} samples; {len(spikes)} spikes were injected")
print(f"largest |raw - median-filtered| at the spikes    : {np.abs(resid[spikes]).min():.2f}"
      f" to {np.abs(resid[spikes]).max():.2f} nN")
print(f"typical |raw - median-filtered| elsewhere (95th %): "
      f"{np.percentile(np.abs(np.delete(resid, spikes)), 95):.3f} nN")

# moving average for comparison (review question 3)
mov = pd.Series(raw).rolling(5, center=True, min_periods=1).mean().values
worst = spikes[np.argmax(np.abs(resid[spikes]))]
print(f"at the largest spike: raw {raw[worst]:.2f}, median {clean[worst]:.2f}, "
      f"moving average {mov[worst]:.2f} nN -> the average smears the spike, the median removes it")

slope = np.polyfit(afm['time_s'], afm['force_clean'], 1)[0]
print(f"straight line removed by detrend: slope {slope:.3f} nN/s, but the drift put in was 0.40 nN/s")
print("  -> over only 3 cycles the slow topography also tilts the fitted line; detrend removes"
      " part of the real signal, so plot before/after and state the choice")

fig, ax = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
ax[0].plot(afm['time_s'], raw, lw=0.5); ax[0].set_ylabel('raw (nN)')
ax[1].plot(afm['time_s'], clean, lw=0.5); ax[1].set_ylabel('median filtered')
ax[2].plot(afm['time_s'], afm['force_detrended'], lw=0.5); ax[2].set_ylabel('detrended')
ax[2].set_xlabel('time (s)')
for a in ax: a.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('outputs/ch03_afm_timeseries.png', dpi=150)

# --- aliasing: 90 Hz sampled at 100 Hz looks exactly like 10 Hz -------------
fs = 100.0
t = np.arange(0, 1, 1 / fs)
x90 = np.cos(2 * np.pi * 90 * t)
x10 = np.cos(2 * np.pi * 10 * t)
print(f"\naliasing: sampled at {fs:.0f} Hz, a 90 Hz and a 10 Hz cosine differ by at most "
      f"{np.abs(x90 - x10).max():.1e} -> identical samples, no filter can separate them")
print("saved outputs/ch03_afm_timeseries.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `04_image_clean.py` — โค้ดที่ 3.4

Ch.3  The three groups of image problems in ssec:image: hot pixels (and why a 5-sigma

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_image_clean.py  จากโฟลเดอร์ของบท
__file__ = '04_image_clean.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  The three groups of image problems in ssec:image: hot pixels (and why a 5-sigma
cut is not enough), noise smoothing, and uneven illumination corrected with a flat field.

Book references: ssec:image, code:image-clean, tab:cleaning-techniques (row "image")
Images are SYNTHETIC (4 x 256 x 256 x 3); image 0, band 0 has two hot pixels.
"""
import os
from pathlib import Path

import numpy as np
from scipy.ndimage import median_filter

os.chdir(Path(__file__).resolve().parent)
images = np.load('data/raw/galaxy_images.npy')
print("images:", images.shape, images.dtype)

### โค้ดที่ 3.4

In [ ]:
# --- code:image-clean -------------------------------------------------------
from scipy.ndimage import gaussian_filter

img = images[0, :, :, 0].astype(float)      # first image, first band
threshold = img.mean() + 5 * img.std()
hot_mask = img > threshold

img_smooth = gaussian_filter(img, sigma=1.0)

# --- what the 5-sigma cut actually found -------------------------------------
ys, xs = np.nonzero(hot_mask)
print(f"threshold = {threshold:.2f}; pixels above it: {hot_mask.sum()}")
near_centre = np.hypot(xs - 128, ys - 128) < 10
print(f"  of these, {near_centre.sum()} are in the galaxy core (real light), "
      f"{(~near_centre).sum()} are elsewhere")

# better test: bright AND very different from the median of its neighbours
local = median_filter(img, size=3)
noise = 1.4826 * np.median(np.abs(img - np.median(img)))       # robust sigma
hot = (img - local) > 10 * noise
print(f"neighbour test (img - local median > 10 sigma): {hot.sum()} pixels at "
      f"{list(zip(*np.nonzero(hot)))}  <- the two injected hot pixels")

fixed = np.where(hot, local, img)
print(f"after replacing them by the local median: max outside the core = "
      f"{fixed[np.hypot(*np.mgrid[0:256, 0:256] - 128) > 20].max():.2f}")
print(f"Gaussian smoothing (sigma = 1 px): noise std {np.std(img[:30, :30]):.3f} -> "
      f"{np.std(img_smooth[:30, :30]):.3f}, peak {img[118:138, 118:138].max():.1f} -> "
      f"{img_smooth[118:138, 118:138].max():.1f} (the core is blurred too)")

# --- third group in the text: vignetting, corrected by dividing by a flat field ----
yy, xx = np.mgrid[0:256, 0:256]
rr = np.hypot(xx - 127.5, yy - 127.5) / 181.0            # 0 at centre, 1 at the corners
vignette = 1 - 0.4 * rr**2                                # corners 40 % darker
observed = fixed * vignette
flat = vignette * (1 + np.random.default_rng(4).normal(0, 0.002, vignette.shape))  # flat-field frame
corrected = observed / flat
corner, centre = (slice(0, 30), slice(0, 30)), (slice(113, 143), slice(113, 143))
print(f"vignetting: corner/centre sky ratio {vignette[corner].mean() / vignette[centre].mean():.2f};"
      f" after dividing by the flat field the image equals the original to within "
      f"{np.median(np.abs(corrected - fixed)[fixed > 2] / fixed[fixed > 2]):.1%} (median)")

## `05_spectrum_afm.py` — โค้ดที่ 3.5

Ch.3  Force-distance curve: remove the baseline, smooth with Savitzky-Golay, and check

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_spectrum_afm.py  จากโฟลเดอร์ของบท
__file__ = '05_spectrum_afm.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Force-distance curve: remove the baseline, smooth with Savitzky-Golay, and check
that smoothing does not destroy the snap-off (adhesion) event.

Book references: ssec:spectrum, code:spectrum-clean
The curve is SYNTHETIC: baseline offset 0.8 nN, contact slope 0.25 nN/nm, adhesion 3.0 nN.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent)
afm_spec = pd.read_csv('data/raw/afm_spectrum.csv')

### โค้ดที่ 3.5

In [ ]:
# --- code:spectrum-clean ----------------------------------------------------
from scipy.signal import savgol_filter

# baseline: mean force where the tip is far from the surface
far_mask = afm_spec['distance_nm'] > 200
baseline = afm_spec.loc[far_mask, 'force_nN'].mean()
afm_spec['force_corrected'] = afm_spec['force_nN'] - baseline

afm_spec['force_smooth'] = savgol_filter(
    afm_spec['force_corrected'].values,
    window_length=11, polyorder=3
)

# --- checks -----------------------------------------------------------------
print(f"baseline from distance > 200 nm: {baseline:.3f} nN (0.8 nN was put in)")
ret = afm_spec['segment'] == 'retract'
print(f"adhesion (minimum of retract curve), true value 3.0 nN:")
print(f"  corrected, not smoothed       : {-afm_spec.loc[ret, 'force_corrected'].min():.2f} nN")
print(f"  Savitzky-Golay, window 11     : {-afm_spec.loc[ret, 'force_smooth'].min():.2f} nN")
for w in (11, 31, 61):
    ma = afm_spec.loc[ret, 'force_corrected'].rolling(w, center=True, min_periods=1).mean()
    sg = savgol_filter(afm_spec.loc[ret, 'force_corrected'].values, w, 3)
    print(f"  window {w:2d}: moving average {-ma.min():.2f} nN, Savitzky-Golay {-sg.min():.2f} nN")
print("-> wide windows flatten the sudden snap-off; Savitzky-Golay keeps it better"
      " than a moving average of the same width")

fig, ax = plt.subplots(figsize=(7, 4))
r = afm_spec[ret]
ax.plot(r['distance_nm'], r['force_corrected'], '.', ms=2, label='corrected')
ax.plot(r['distance_nm'], r['force_smooth'], lw=1.2, label='Savitzky-Golay (11, 3)')
ax.plot(r['distance_nm'], savgol_filter(r['force_corrected'].values, 61, 3), lw=1.2,
        label='Savitzky-Golay (61, 3): too wide')
ax.set_xlim(0, 80); ax.set_xlabel('distance (nm)'); ax.set_ylabel('force (nN)')
ax.set_title('retract curve: before and after smoothing'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('outputs/ch03_afm_smoothing.png', dpi=150)
print("saved outputs/ch03_afm_smoothing.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `06_pandas_basics.py` — โค้ดที่ 3.6, 3.7, 3.8, 3.9, 3.10

Ch.3  pandas basics on the (SYNTHETIC) Galaxy10-style table: load, check dtypes,

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_pandas_basics.py  จากโฟลเดอร์ของบท
__file__ = '06_pandas_basics.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  pandas basics on the (SYNTHETIC) Galaxy10-style table: load, check dtypes,
select, filter, group.

Book references: sec:pandas, code:df-load, code:dtype-check, code:df-select,
                 code:df-filter, code:groupby
"""
import os
from pathlib import Path

os.chdir(Path(__file__).resolve().parent)

### โค้ดที่ 3.6

In [ ]:
# --- code:df-load -----------------------------------------------------------
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/galaxy10_metadata.csv', index_col='image_id')

print(df.shape)
print(df.dtypes)             # check the type of every column
print(df.head())

### โค้ดที่ 3.7

In [ ]:
# --- code:dtype-check -------------------------------------------------------
expected_numeric = ['redshift', 'r_mag', 'g_mag']
for col in expected_numeric:
    if not pd.api.types.is_numeric_dtype(df[col]):
        print(f"{col} dtype = {df[col].dtype} (expected numeric)")

# why? a few entries are text
bad = pd.to_numeric(df['g_mag'], errors='coerce').isna() & df['g_mag'].notna()
print(f"g_mag entries that are not numbers: {df.loc[bad, 'g_mag'].unique().tolist()} "
      f"in {bad.sum()} rows -> convert with pd.to_numeric(errors='coerce')")
df['g_mag'] = pd.to_numeric(df['g_mag'], errors='coerce')

### โค้ดที่ 3.8

In [ ]:
# --- code:df-select ---------------------------------------------------------
# one column -> Series
redshifts = df['redshift']

# several columns -> DataFrame
photometry = df[['r_mag', 'g_mag']]

# rows by label (loc) or by position (iloc)
row_galaxy_42 = df.loc['galaxy_00042']
row_first_10  = df.iloc[:10]
print(f"\ntypes: {type(redshifts).__name__}, {type(photometry).__name__}, "
      f"{type(row_galaxy_42).__name__}, {type(row_first_10).__name__} {row_first_10.shape}")
print(row_galaxy_42.to_dict())

### โค้ดที่ 3.9

In [ ]:
# --- code:df-filter ---------------------------------------------------------
z_min, z_max = 0.05, 0.15
mask = df['redshift'].notna() & df['redshift'].between(z_min, z_max)
low_z = df[mask]
print(f"\n{len(low_z)} galaxies with {z_min} <= z <= {z_max}")
print(f"between() alone already drops NaN: {df['redshift'].between(z_min, z_max).sum()} "
      "-> the notna() makes the intention explicit")

### โค้ดที่ 3.10

In [ ]:
# --- code:groupby -----------------------------------------------------------
summary = df.groupby('galaxy_class').agg(
    count      = ('redshift', 'size'),
    n_z        = ('redshift', 'count'),     # non-missing only
    mean_z     = ('redshift', 'mean'),
    median_z   = ('redshift', 'median'),
    std_z      = ('redshift', 'std'),
    mean_r_mag = ('r_mag', 'mean'),
)
pd.set_option('display.width', 120)
print("\n", summary.round(3))
print("\n'size' counts rows, 'count' counts non-missing values: "
      f"{summary['count'].sum()} rows, {summary['n_z'].sum()} redshifts")

## `07_imputation.py` — โค้ดที่ 3.11

Ch.3  Three ways to fill missing redshifts and what each one does to the mean and spread.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 07_imputation.py  จากโฟลเดอร์ของบท
__file__ = '07_imputation.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Three ways to fill missing redshifts and what each one does to the mean and spread.

Book references: ssec:missing, tab:imputation, code:imputation, misconception box on mean imputation
Because the data are SYNTHETIC, the true redshift of every galaxy is known
(data/raw/galaxy10_truth.csv), so each method can be compared with the truth.
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent)
df = pd.read_csv('data/raw/galaxy10_metadata.csv', index_col='image_id')
truth = pd.read_csv('data/raw/galaxy10_truth.csv', index_col='image_id')
df = df[df['r_mag'] >= 8]                       # drop the 5 foreground stars first
truth = truth.loc[df.index]

### โค้ดที่ 3.11

In [ ]:
# --- code:imputation --------------------------------------------------------
df['z_imp_mean']   = df['redshift'].fillna(df['redshift'].mean())
df['z_imp_median'] = df['redshift'].fillna(df['redshift'].median())

# toy photometric-redshift relation (for teaching only)
def photo_z_estimate(r_mag):
    return 0.05 * (r_mag - 15.0)

df['z_imp_photo'] = df['redshift'].fillna(photo_z_estimate(df['r_mag']))

# --- compare ----------------------------------------------------------------
miss = df['redshift'].isna()
rows = {
    'truth (all galaxies)':   truth['z_true'],
    'drop missing rows':      df['redshift'].dropna(),
    'fill with mean':         df['z_imp_mean'],
    'fill with median':       df['z_imp_median'],
    'fill from r_mag':        df['z_imp_photo'],
}
print(f"{len(df)} galaxies, {miss.sum()} missing redshifts ({100 * miss.mean():.1f} %)\n")
print(f"{'method':24s} {'N':>5s} {'mean z':>8s} {'std z':>8s}")
for name, z in rows.items():
    print(f"{name:24s} {z.size:5d} {z.mean():8.4f} {z.std():8.4f}")
err = (df.loc[miss, 'z_imp_photo'] - truth.loc[miss, 'z_true'])
print(f"\nfilled values only: mean-fill error {np.mean(df.loc[miss, 'z_imp_mean'] - truth.loc[miss, 'z_true']):+.3f}, "
      f"photo-z error {err.mean():+.3f} +- {err.std():.3f}")
print("in this synthetic world the toy relation is right by construction; with a wrong"
      " relation its error would enter every filled value")

## `08_merge.py` — โค้ดที่ 3.12

Ch.3  Joining two tables safely: check keys, then compare row counts.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 08_merge.py  จากโฟลเดอร์ของบท
__file__ = '08_merge.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Joining two tables safely: check keys, then compare row counts.

Book references: ssec:merge, tab:join, code:merge, review question 5 (10,000 x 10,000 -> 10,250)
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent)

### โค้ดที่ 3.12

In [ ]:
# --- code:merge -------------------------------------------------------------
metadata    = pd.read_csv('data/raw/galaxy10_metadata.csv', index_col='image_id')
image_paths = pd.read_csv('data/raw/galaxy10_image_index.csv', index_col='image_id')

assert metadata.index.is_unique, "duplicate image_id in metadata"
assert image_paths.index.is_unique, "duplicate image_id in image_paths"

combined = metadata.merge(image_paths,
                          left_index=True, right_index=True, how='inner')
print(len(metadata), len(image_paths), len(combined))

for how in ('inner', 'left', 'right', 'outer'):
    n = len(metadata.merge(image_paths, left_index=True, right_index=True, how=how))
    print(f"how='{how:5s}': {n} rows")
print("-> 20 galaxies have no image and 15 images have no metadata row; 'inner' drops both"
      " silently, 'outer' shows them")

# --- review question 5: 10,000 x 10,000 rows -> 10,250 rows -------------------
# rows of an inner join = sum over keys of n_A(key) * n_B(key).
# If A had unique keys, every row of B could match at most one row of A, so the result
# could never exceed len(B) = 10,000. Getting 10,250 therefore needs duplicates in BOTH.
rng = np.random.default_rng(5)
dup = rng.choice(10_000, 125, replace=False)             # 125 keys recorded twice ...
single = np.setdiff1d(np.arange(10_000), dup)[:9_750]
keys = np.concatenate([single, np.repeat(dup, 2)])      # 9,750 + 250 = 10,000 rows
A = pd.DataFrame({'key': keys, 'x': rng.normal(size=keys.size)})
B = pd.DataFrame({'key': keys, 'y': rng.normal(size=keys.size)})   # ... in both tables
print(f"\nA: {len(A)} rows, B: {len(B)} rows, 125 keys appear twice in each")
print(f"inner join: {len(A.merge(B, on='key', how='inner')):,} rows  (9,750 + 125 x 2 x 2)")
B1 = B.drop_duplicates('key')
print(f"with duplicates only in A (B made unique, {len(B1)} rows): "
      f"{len(A.merge(B1, on='key', how='inner')):,} rows -> never more than len(A)")

## `09_units_pint.py` — โค้ดที่ 3.13

Ch.3  Carry units through a calculation with pint; unit mistakes fail loudly.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 09_units_pint.py  จากโฟลเดอร์ของบท
__file__ = '09_units_pint.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Carry units through a calculation with pint; unit mistakes fail loudly.

Book references: ssec:pint, code:pint (Mars Climate Orbiter: 1 lbf s = 4.45 N s)
Needs: pip install pint
"""

### โค้ดที่ 3.13

In [ ]:
# --- code:pint --------------------------------------------------------------
import pint
import scipy.constants as const

ureg = pint.UnitRegistry()

v = 5.0 * ureg.meter / ureg.second
m = 2.0 * ureg.kilogram
KE = 0.5 * m * v**2
print(KE.to(ureg.joule))           # 25.0 joule

# attach units to a scipy constant
g = const.g * ureg.meter / ureg.second**2
h = 10.0 * ureg.meter
t_fall = (2 * h / g)**0.5
print(t_fall.to(ureg.second))      # about 1.428 second

# a unit mistake is caught immediately
try:
    bad = m + h                     # kg + m
except pint.errors.DimensionalityError as e:
    print(f"caught: {e}")

# --- the Mars Climate Orbiter factor ----------------------------------------
impulse = 1.0 * ureg.pound_force * ureg.second
print(f"\n1 lbf s = {impulse.to(ureg.newton * ureg.second):.4f}  -> the factor of about 4.45 in the text")

# --- AFM units used in this chapter -----------------------------------------
k = 0.1 * ureg.nanonewton / ureg.nanometer
print(f"0.1 nN/nm = {k.to(ureg.newton / ureg.meter)}")
work = (1 * ureg.nanonewton * ureg.nanometer).to(ureg.joule)
print(f"1 nN x 1 nm = {work:.0e} = {work.to(ureg.attojoule):.3f}")

## `10_reynolds.py` — โค้ดที่ 3.14

Ch.3  Buckingham pi in practice: the Reynolds number as one feature replacing three.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 10_reynolds.py  จากโฟลเดอร์ของบท
__file__ = '10_reynolds.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Buckingham pi in practice: the Reynolds number as one feature replacing three.

Book references: ssec:dim-analysis, ex:reynolds, code:reynolds
Check of the text: sorted by Re the two flow classes separate with one threshold,
while v alone or L alone does NOT separate them.
"""
import pandas as pd

### โค้ดที่ 3.14

In [ ]:
# --- code:reynolds ----------------------------------------------------------
flow_df = pd.DataFrame({
    'v_ms':       [0.005, 0.10, 0.02, 0.015, 0.08],
    'L_m':        [0.10,  0.01, 0.50, 0.02,  0.08],
    'nu_m2s':     [1e-6] * 5,
    'flow_class': ['laminar', 'laminar', 'turbulent', 'laminar', 'turbulent'],
})
flow_df['Re'] = flow_df['v_ms'] * flow_df['L_m'] / flow_df['nu_m2s']
# Re = 500, 1000, 10000, 300, 6400

print(flow_df.to_string(index=False))


def separable(col):
    lam = flow_df.loc[flow_df.flow_class == 'laminar', col]
    tur = flow_df.loc[flow_df.flow_class == 'turbulent', col]
    return lam.max() < tur.min() or tur.max() < lam.min()


for col in ('v_ms', 'L_m', 'Re'):
    print(f"one threshold on {col:5s} separates the classes: {separable(col)}")

## `11_invariants.py` — โค้ดที่ 3.15

Ch.3  Rotation-invariant image features, and why '==' is the wrong test for floats.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 11_invariants.py  จากโฟลเดอร์ของบท
__file__ = '11_invariants.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Rotation-invariant image features, and why '==' is the wrong test for floats.

Book references: ssec:symmetry-feat, tab:invariant, code:invariant,
                 misconception box "invariant values must agree to every decimal"
"""
import numpy as np

### โค้ดที่ 3.15

In [ ]:
# --- code:invariant ---------------------------------------------------------
def total_flux(image):
    return image.sum()

def radial_profile(image, center=None):
    if center is None:
        center = ((image.shape[0] - 1) / 2, (image.shape[1] - 1) / 2)
    y, x = np.indices(image.shape)
    r = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)
    profile = np.bincount(r.ravel(), weights=image.ravel())
    counts  = np.bincount(r.ravel())
    return profile / np.maximum(counts, 1)

img     = np.random.default_rng(2026).normal(size=(64, 64))
img_rot = np.rot90(img)
print(np.isclose(total_flux(img), total_flux(img_rot)))   # True
print(np.allclose(radial_profile(img), radial_profile(img_rot)))   # True

# --- the misconception box --------------------------------------------------
print(f"total_flux(img) == total_flux(img_rot): {total_flux(img) == total_flux(img_rot)}")
print(f"difference: {total_flux(img) - total_flux(img_rot):.1e}  (order of the additions differs)")

# why the default centre is ((N-1)/2, (N-1)/2): rot90 turns the pixel grid about that
# point; for a 64 x 64 image it is (31.5, 31.5), not the pixel (32, 32)
c = (np.array(img.shape) - 1) / 2
prof, prof_rot = radial_profile(img), radial_profile(img_rot)
print(f"radial profile about ({c[0]}, {c[1]}): max |difference| after rotation = "
      f"{np.abs(prof - prof_rot).max():.1e}")
p32, p32r = radial_profile(img, center=(32, 32)), radial_profile(img_rot, center=(32, 32))
print(f"radial profile about the pixel (32, 32): max |difference| = "
      f"{np.abs(p32 - p32r).max():.2f}  <- not invariant: wrong rotation centre")

# first moment is NOT invariant
y, x = np.indices(img.shape)
print(f"first moment sum(x*I): {np.sum(x * img):.2f} -> after rotation {np.sum(x * img_rot):.2f}")

# concentration and asymmetry on a smooth synthetic galaxy
r = np.hypot(x - c[1], y - c[0])
gal = np.exp(-r / 6.0)
def radius_containing(image, frac):
    rr = np.hypot(*(np.indices(image.shape) - c[:, None, None]))
    order = np.argsort(rr.ravel())
    cum = np.cumsum(image.ravel()[order]) / image.sum()
    return rr.ravel()[order][np.searchsorted(cum, frac)]
C = 5 * np.log10(radius_containing(gal, 0.8) / radius_containing(gal, 0.2))
A = np.abs(gal - np.rot90(gal, 2)).sum() / np.abs(gal).sum()
print(f"exponential disk: concentration C = {C:.2f}, asymmetry A = {A:.1e}")

## `12_invariant_mass.py` — โค้ดที่ 3.16

Ch.3  Conservation law as a feature: four-lepton invariant mass (Higgs search).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 12_invariant_mass.py  จากโฟลเดอร์ของบท
__file__ = '12_invariant_mass.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Conservation law as a feature: four-lepton invariant mass (Higgs search).

Book references: ssec:conservation-feat, ex:invariant-mass, code:invmass
Events are SYNTHETIC: 400 H -> ZZ* -> 4l decays (m_H = 125 GeV, 1.8 GeV resolution)
plus 1,600 smooth background events, all boosted along the beam. The individual lepton
energies change from event to event with the boost; the invariant mass does not.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent)
events = pd.read_csv('data/raw/higgs_4l_events.csv')

### โค้ดที่ 3.16

In [ ]:
# --- code:invmass -----------------------------------------------------------
E  = events[['E1',  'E2',  'E3',  'E4' ]].sum(axis=1)
px = events[['px1', 'px2', 'px3', 'px4']].sum(axis=1)
py = events[['py1', 'py2', 'py3', 'py4']].sum(axis=1)
pz = events[['pz1', 'pz2', 'pz3', 'pz4']].sum(axis=1)

m2 = E**2 - px**2 - py**2 - pz**2
events['M_inv'] = np.sqrt(np.clip(m2, 0, None))   # guard tiny negatives

# --- checks -----------------------------------------------------------------
sig = events['is_signal']
print(f"{len(events)} events ({sig.sum()} signal)")
print(f"signal M_inv: median {events.loc[sig, 'M_inv'].median():.2f} GeV, "
      f"std {events.loc[sig, 'M_inv'].std():.2f} GeV  (generated: 125 GeV, 1.8 GeV)")
print(f"total energy E of signal events ranges {E[sig].min():.0f}-{E[sig].max():.0f} GeV "
      "-> E alone is not a good feature; M_inv is")
print(f"events with m2 < 0 before clipping: {(m2 < 0).sum()}")
win = events['M_inv'].between(120, 130)
print(f"window 120-130 GeV: {win.sum()} events, {100 * events.loc[win, 'is_signal'].mean():.0f} % signal")
pt1 = np.hypot(events['px1'], events['py1'])
print(f"transverse momentum of lepton 1: median {pt1.median():.1f} GeV (another conserved-quantity feature)")

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.arange(70, 300, 2.5)
ax.hist(events['M_inv'], bins=bins, histtype='step', color='k', label='all events')
ax.hist(events.loc[~sig, 'M_inv'], bins=bins, histtype='stepfilled', alpha=0.3,
        label='background only')
ax.set_xlabel(r'four-lepton invariant mass $M_{4\ell}$ [GeV]'); ax.set_ylabel('events / 2.5 GeV')
ax.set_title('SYNTHETIC H -> 4 leptons'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('outputs/ch03_invariant_mass.png', dpi=150)
print("saved outputs/ch03_invariant_mass.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `13_afm_normalize.py` — โค้ดที่ 3.17

Ch.3  Physics-aware normalisation: turn AFM force back into cantilever deflection.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 13_afm_normalize.py  จากโฟลเดอร์ของบท
__file__ = '13_afm_normalize.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  Physics-aware normalisation: turn AFM force back into cantilever deflection.

Book references: ssec:phys-normalize, ex:afm-norm, code:afm-norm, review question 7
The curve is SYNTHETIC. The script also shows the text's point: dividing by the SAME k
that produced the forces gives back the measured deflection exactly, while a different k
scales every point by the ratio of the two values (a systematic, not random, error).
"""
import os
from pathlib import Path

import numpy as np

os.chdir(Path(__file__).resolve().parent)

### โค้ดที่ 3.17

In [ ]:
# --- code:afm-norm ----------------------------------------------------------
import pandas as pd
afm = pd.read_csv('data/raw/afm_spectrum.csv')
k_cantilever = 0.1        # N/m, identical to nN/nm
afm['deflection_nm'] = afm['force_nN'] / k_cantilever

print(afm[['segment', 'distance_nm', 'force_nN', 'deflection_nm']].iloc[[0, 250, 300, 560]]
      .to_string(index=False))

# suppose the instrument measured deflection d and computed F = k_used * d
d_measured = afm['deflection_nm'].values
for k_used in (0.1, 0.12):
    F = k_used * d_measured
    d_back = F / k_cantilever
    print(f"forces made with k = {k_used} N/m, divided by k = {k_cantilever} N/m: "
          f"deflection off by a factor {np.median(d_back / d_measured):.2f} at every point")

# a calibration error in k is systematic: averaging more points does not remove it
rng = np.random.default_rng(3)
d0 = 5.0                                   # nm, true deflection of a repeated measurement
k_true = 0.1                               # N/m
k_cal = 0.11                               # N/m, one calibration that happens to be 10 % high
print("mean force from N repeated measurements (true value "
      f"{k_true * d0:.3f} nN), deflection noise 0.5 nm:")
for n in (10, 1_000, 100_000):
    d = d0 + rng.normal(0, 0.5, n)
    print(f"  N = {n:6d}: {np.mean(k_cal * d):.4f} nN")
print("-> the random part shrinks with N, the 10 % from k stays: a systematic uncertainty")

## `14_scorecard.py` — โค้ดที่ 3.18

Ch.3  The data-quality scorecard of code:scorecard as reusable code (src/scorecard.py).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 14_scorecard.py  จากโฟลเดอร์ของบท
__file__ = '14_scorecard.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.3  The data-quality scorecard of code:scorecard as reusable code (src/scorecard.py).

Book references: ssec:scorecard, code:scorecard, tab:quality-4
Prints the empty template (same items and maxima as the book: 9 + 9 + 6 + 3 + 9 = 36).
A filled-in example for the Galaxy10-style table is in exercises/E3_7_scorecard.py.
"""
import os
import sys
from pathlib import Path

os.chdir(Path(__file__).resolve().parent)
sys.path.insert(0, ".")
from src.scorecard import ITEMS, Scorecard

print(Scorecard("<dataset name>").report())
print(f"\nsections: {', '.join(f'{k} /{3 * len(v)}' for k, v in ITEMS.items())}")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E3_1_outlier_cut.py`

E3.1  What happens when 50 'abnormal' high-energy events are cut without checking.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E3_1_outlier_cut.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E3_1_outlier_cut.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E3.1  What happens when 50 'abnormal' high-energy events are cut without checking.

A SYNTHETIC toy: 950 events follow the theory (exponential energy spectrum, mean 10 GeV)
and 50 come from a real extra process (a peak at 45 GeV) that the theory does not include.
Cutting the 50 makes the data 'agree with theory' -- and throws away the new physics.
"""
import numpy as np
from scipy import stats

rng = np.random.default_rng(31)
theory_mean = 10.0
E = np.concatenate([rng.exponential(theory_mean, 950), rng.normal(45, 2, 50)])

cut = np.sort(E)[:-50]                              # remove the 50 highest energies
for name, x in (("all 1000 events", E), ("after cutting 50", cut)):
    ks = stats.kstest(x, "expon", args=(0, theory_mean))
    print(f"{name:17s}: mean {x.mean():6.2f} GeV, std {x.std(ddof=1):6.2f} GeV, "
          f"KS test vs theory p = {ks.pvalue:.3g}")
print(f"theory: mean = std = {theory_mean} GeV")
removed = np.sort(E)[-50:]
print(f"the 50 removed events: {removed.min():.1f}-{removed.max():.1f} GeV, "
      f"{np.mean(np.abs(removed - 45) < 6):.0%} of them within 6 GeV of 45 GeV "
      "-> a peak, not random noise")
print("the 'agreement' after the cut was produced by the cut itself (circular reasoning)")

## `exercises/E3_2_clean_messy.py`

E3.2  Clean the messy Galaxy10-style table (SYNTHETIC, ~500 rows).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E3_2_clean_messy.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E3_2_clean_messy.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E3.2  Clean the messy Galaxy10-style table (SYNTHETIC, ~500 rows).

Writes data/processed/galaxy10_cleaned.csv and prints every decision with its count.
Rule of the book: MARK problems with flag columns, do not delete rows.
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent.parent)

# 1. look before converting: the sentinel is invisible to isna()
raw = pd.read_csv('data/raw/galaxy10_messy.csv')
print("isna() before handling the sentinel:", raw.isna().sum().to_dict())
print(raw.describe().loc[['min', 'mean', 'max']].round(3))
print(f"mean redshift with -999 left in: {raw['redshift'].mean():.2f}  <- negative, meaningless\n")

# 2. load again, telling pandas that -999 means 'not measured'
df = pd.read_csv('data/raw/galaxy10_messy.csv', na_values=[-999])
n_sent = (raw[['redshift', 'r_mag']] == -999).sum()
print(f"sentinels turned into NaN: redshift {n_sent['redshift']}, r_mag {n_sent['r_mag']}")

# 3. flags instead of deletion
df['flag_no_redshift'] = df['redshift'].isna()
df['flag_negative_z']  = df['redshift'] < 0            # physically impossible here
df['flag_r_range']     = df['r_mag'].notna() & ~df['r_mag'].between(8, 25)
df['flag_no_r_mag']    = df['r_mag'].isna()
df['usable'] = ~(df[['flag_no_redshift', 'flag_negative_z', 'flag_r_range',
                     'flag_no_r_mag']].any(axis=1))
for c in [c for c in df.columns if c.startswith('flag_')]:
    print(f"  {c:18s}: {df[c].sum():3d} rows")
print(f"  usable for redshift work: {df['usable'].sum()} of {len(df)}")

out = Path('data/processed'); out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / 'galaxy10_cleaned.csv', index=False)
print(f"\nsaved data/processed/galaxy10_cleaned.csv ({len(df)} rows, nothing deleted)")
print(f"mean redshift of usable rows: {df.loc[df['usable'], 'redshift'].mean():.4f}")

## `exercises/E3_3_groupby.py`

E3.3  Summarise r_mag and redshift by galaxy class; plot the mean with the STANDARD

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E3_3_groupby.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E3_3_groupby.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E3.3  Summarise r_mag and redshift by galaxy class; plot the mean with the STANDARD
ERROR of the mean (std / sqrt(n)), not the standard deviation.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent.parent)
df = pd.read_csv('data/processed/galaxy10_cleaned.csv')
use = df[df['usable']]
s = use.groupby('galaxy_class').agg(n_rows=('redshift', 'size'), n_z=('redshift', 'count'),
                                     mean_z=('redshift', 'mean'), std_z=('redshift', 'std'),
                                     mean_r=('r_mag', 'mean'))
s['sem_z'] = s['std_z'] / np.sqrt(s['n_z'])
miss = df.groupby('galaxy_class')['flag_no_redshift'].mean().rename('frac_missing_z')
s = s.join(miss).sort_values('mean_z')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
print(s.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(range(len(s)), s['mean_z'], yerr=s['sem_z'], fmt='o', capsize=3)
ax.set_xticks(range(len(s))); ax.set_xticklabels(s.index, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('mean redshift ± SEM'); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig('outputs/E3_3_mean_redshift_by_class.png', dpi=150)
print("\nsaved outputs/E3_3_mean_redshift_by_class.png")
print("with ~50 galaxies per class the SEM is ~0.006; differences between classes of that"
      " size are not evidence of a physical difference. In this synthetic table the classes"
      " were drawn with the same redshift distribution, so none is expected.")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E3_4_imputation_sensitivity.py`

E3.4  Sensitivity of the mean redshift to the missing-value method, against the truth

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E3_4_imputation_sensitivity.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E3_4_imputation_sensitivity.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E3.4  Sensitivity of the mean redshift to the missing-value method, against the truth
(known because the data are SYNTHETIC). Also: what if the photo-z relation is wrong?
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent.parent)
df = pd.read_csv('data/raw/galaxy10_metadata.csv', index_col='image_id')
truth = pd.read_csv('data/raw/galaxy10_truth.csv', index_col='image_id')
df = df[df['r_mag'] >= 8]
z_true = truth.loc[df.index, 'z_true']

good = lambda r: 0.05 * (r - 15.0)            # the book's toy relation
wrong = lambda r: 0.04 * (r - 15.0)           # a relation with a 20 % wrong slope
methods = {
    'truth':                      z_true,
    'drop rows':                  df['redshift'].dropna(),
    'fill median':                df['redshift'].fillna(df['redshift'].median()),
    'fill from r_mag (right)':    df['redshift'].fillna(good(df['r_mag'])),
    'fill from r_mag (wrong)':    df['redshift'].fillna(wrong(df['r_mag'])),
}
print(f"{'method':28s} {'mean z':>8s} {'std z':>8s} {'bias of mean':>13s}")
for k, z in methods.items():
    print(f"{k:28s} {z.mean():8.4f} {z.std():8.4f} {z.mean() - z_true.mean():+13.4f}")
print("\nprediction check: dropping rows lowers the mean (faint = far galaxies removed); "
      "median filling also shrinks the spread; the relation-based fill is best only if the "
      "relation is right.")

## `exercises/E3_5_afm_features.py`

E3.5  Three physical features from 15 SYNTHETIC force-distance curves, with units,

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E3_5_afm_features.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E3_5_afm_features.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E3.5  Three physical features from 15 SYNTHETIC force-distance curves, with units,
compared with the values used to generate the curves. Writes data/processed/afm_features.csv.
"""
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

os.chdir(Path(__file__).resolve().parent.parent)
sys.path.insert(0, ".")
from src.features import (adhesion_work_aJ, contact_slope_nN_per_nm, max_adhesion_nN,
                          remove_baseline)

curves = pd.read_csv('data/raw/afm_curves.csv')
truth = pd.read_csv('data/raw/afm_curves_truth.csv', index_col='curve_id')
rows = []
for cid, c in curves.groupby('curve_id'):
    c, base = remove_baseline(c)
    rows.append({'curve_id': cid, 'baseline_nN': base,
                 'slope_nN_per_nm': contact_slope_nN_per_nm(c),
                 'adhesion_nN': max_adhesion_nN(c),
                 'adhesion_nN_sg31': max_adhesion_nN(c, window=31),
                 'work_aJ': adhesion_work_aJ(c)})
feat = pd.DataFrame(rows).set_index('curve_id')
Path('data/processed').mkdir(parents=True, exist_ok=True)
feat.round(4).to_csv('data/processed/afm_features.csv')

for col in ('baseline_nN', 'slope_nN_per_nm', 'adhesion_nN', 'work_aJ'):
    err = feat[col] - truth[col]
    print(f"{col:17s}: mean error {err.mean():+.4f}, rms {np.sqrt((err**2).mean()):.4f}")
err_sg = feat['adhesion_nN_sg31'] - truth['adhesion_nN']
print(f"adhesion after Savitzky-Golay (31): mean error {err_sg.mean():+.3f} nN "
      "-> smoothing biases the peak adhesion low")
extra = 0.5 * truth['adhesion_nN'] * 1.0          # half a 1 nm step at the snap-off
print(f"work: the trapezoid rule joins the last stuck point to the first free point, adding"
      f" about 0.5 x adhesion x 1 nm = {extra.mean():.2f} aJ on average -> sampling limits this feature")
print("units: slope nN/nm = N/m; adhesion nN; work nN*nm = 1e-18 J = aJ")
print("saved data/processed/afm_features.csv")

## `exercises/E3_7_scorecard.py`

E3.7  Fill in the scorecard for the cleaned Galaxy10-style table, with evidence that is

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E3_7_scorecard.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E3_7_scorecard.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E3.7  Fill in the scorecard for the cleaned Galaxy10-style table, with evidence that is
computed, not typed. Scores that need human judgement are marked as such.
"""
import os
import sys
from pathlib import Path

import pandas as pd

os.chdir(Path(__file__).resolve().parent.parent)
sys.path.insert(0, ".")
from src.scorecard import Scorecard

df = pd.read_csv('data/processed/galaxy10_cleaned.csv')
sc = Scorecard("data/processed/galaxy10_cleaned.csv (SYNTHETIC)", reviewer="example")
miss = df['flag_no_redshift'].mean()
fz = df.groupby(pd.cut(df['r_mag'], [0, 17, 18, 19, 30]), observed=True)['flag_no_redshift'].mean()
sc.score("completeness", "missing values: type and cause identified", 3,
         f"{miss:.0%} missing z; rises with r_mag: {fz.round(2).tolist()} -> MAR")
sc.score("completeness", "sample selection bias assessed", 2,
         "faint galaxies under-represented after dropping; not corrected")
sc.score("completeness", "coverage of the parameter space", 1,
         f"r_mag {df['r_mag'].min():.1f}-{df['r_mag'].max():.1f} incl. flagged values")
sc.score("accuracy", "calibration source and date", 0, "synthetic table: none")
sc.score("accuracy", "systematic uncertainty estimated", 1, "only imputation sensitivity (E3.4)")
sc.score("accuracy", "at least one sanity check against known physics", 2,
         f"negative z flagged: {df['flag_negative_z'].sum()} rows")
sc.score("consistency", "units stated and consistent for every column", 2,
         "z dimensionless, r_mag mag; not stored in the file")
sc.score("consistency", "dtypes and naming consistent", 3,
         f"all numeric: {all(pd.api.types.is_numeric_dtype(df[c]) for c in ['redshift', 'r_mag', 'g_mag'])}")
sc.score("timeliness", "dataset version and last-update date", 1, "generator seed only")
sc.score("reproducibility", "cleaning script and seed committed to git", 3, "E3_2_clean_messy.py, SEED = 2026")
sc.score("reproducibility", "requirements.txt and environment documented", 3, "code/requirements.txt")
sc.score("reproducibility", "notebook passes Restart & Run All", 0, "scripts, no notebook")
print(sc.report())
print("\nweak points: (1) redshift missing not at random -> any mean z is biased (Ch.4);"
      " (2) no calibration information -> systematic errors cannot be quoted.")

# ---- second data set: AFM features from E3.5 --------------------------------------
feat = pd.read_csv('data/processed/afm_features.csv')
af = Scorecard("data/processed/afm_features.csv (SYNTHETIC)", reviewer="example")
af.score("completeness", "missing values: type and cause identified", 3,
         f"no missing values in {len(feat)} curves")
af.score("completeness", "sample selection bias assessed", 1, "15 curves, selection not described")
af.score("completeness", "coverage of the parameter space", 2,
         f"slope {feat.slope_nN_per_nm.min():.2f}-{feat.slope_nN_per_nm.max():.2f} nN/nm")
af.score("accuracy", "calibration source and date", 0, "cantilever k not calibrated (synthetic)")
af.score("accuracy", "systematic uncertainty estimated", 2,
         f"smoothing bias of adhesion: {(feat.adhesion_nN_sg31 - feat.adhesion_nN).mean():+.2f} nN")
af.score("accuracy", "at least one sanity check against known physics", 3,
         "work ~ 0.5 x adhesion x snap length (triangle)")
af.score("consistency", "units stated and consistent for every column", 3, "units in column names")
af.score("consistency", "dtypes and naming consistent", 3, "all float")
af.score("timeliness", "dataset version and last-update date", 1, "generator seed only")
af.score("reproducibility", "cleaning script and seed committed to git", 3, "E3_5_afm_features.py, src/features.py")
af.score("reproducibility", "requirements.txt and environment documented", 3, "code/requirements.txt")
af.score("reproducibility", "notebook passes Restart & Run All", 3, "notebooks/04_quality_scorecard.ipynb")
print("\n" + af.report())
print("\nweak points: (1) the spring constant k is not calibrated -> every force carries one"
      " common systematic error; (2) peak adhesion depends on the smoothing window.")